In [ ]:
!pip install -q transformers accelerate bitsandbytes huggingface_hub

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from huggingface_hub import login

login()  # paste your HF token

model_id = "Qwen/Qwen3.5-4B"
tokenizer = AutoTokenizer.from_pretrained(model_id)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.4 MB/s eta 0:00:00


config.json:   0%|          | 0.00/3.16k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/6.72M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/3.35M [00:00<?, ?B/s]

...

chat_template.jinja:   0%|          | 0.00/7.76k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/76.2k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

#SINHALA LANGUAGE

In [ ]:
import json

DATA_PATH = "/content/sample_data/sri_lankan_simplified.json"

with open(DATA_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Loaded {len(data)} items")

Loaded 203 items


In [ ]:
INSTRUCTION_PROMPT = (
    """ You are a Sinhala Expert for answering multiple-choice questions.
    You need to always evaluate these questions based on the Sri Lankan context and provide the most accurate answer.
    You should only respond with the correct option text, without any additional explanation or commentary.

Question: {question}
Option A: {option A}
Option B: {option B}
Option C: පිළිතුරු දෙකම නිවැරදියි.
Option D: පිළිතුරු දෙකම නිවැරදි නොවේ.
Return only the correct option text. Expected output is either the text of 'A', 'B', 'C', or 'D'.
 """
)

def build_prompt(item):
    lines = [INSTRUCTION_PROMPT, "", item["scenario_question"], "", "Options:"]
    for letter, text in item["options"].items():
        lines.append(f"{letter}. {text}")
    return "\n".join(lines)

In [ ]:
import re
def generate_answer(prompt, valid_letters, max_new_tokens=20):
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True,enable_thinking=False
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)

    input_len = inputs["input_ids"].shape[-1]
    raw = tokenizer.decode(output[0][input_len:], skip_special_tokens=True).strip()

    if "</think>" in raw:
        raw_for_parsing = raw.split("</think>")[-1].strip()
    else:
        raw_for_parsing = raw

    # First try a clean letter match (model followed instructions)
    match = re.search(r"\b([A-D])\b", raw.upper())
    if match and match.group(1) in valid_letters:
        predicted = match.group(1)
    # Fall back: model echoed the option content instead of the letter
    elif "D" in raw:
        predicted = "0"
    elif "C" in raw:   # check this AFTER the "නොවේ" (negative) case
        predicted = "Both"
    else:
        predicted = None

    return predicted, raw

In [ ]:
OUTPUT_PATH = "/content/sri_lankan_results.json"

for i, item in enumerate(data):
    if item.get("model_answer"):  # already processed, skip (resume-safe)
        continue

    prompt = build_prompt(item)
    predicted, raw_output = generate_answer(prompt, list(item["options"].keys()))

    item["model_answer"] = predicted
    item["raw_model_output"] = raw_output

    with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    print(f"[{i+1}/{len(data)}] {item['id']} -> {predicted} (gold: {item['gold_answer']})")

print("All done.")

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


[1/203] SI_2004 -> A (gold: A)
[2/203] SI_2007 -> B (gold: B)
[3/203] SI_2009 -> A (gold: A)
[4/203] SI_2012 -> B (gold: B)
[5/203] SI_2015 -> A (gold: A)
[6/203] SI_2017 -> A (gold: A)
[7/203] SI_2018 -> A (gold: A)
[8/203] SI_2022 -> A (gold: A)
[9/203] SI_2026 -> A (gold: B)
[10/203] SI_2027 -> A (gold: A)
[11/203] SI_2028 -> A (gold: A)
[12/203] SI_2033 -> A (gold: B)
[13/203] SI_2036 -> A (gold: A)
[14/203] SI_2038 -> A (gold: A)
[15/203] SI_2039 -> A (gold: A)
[16/203] SI_2046 -> A (gold: A)
[17/203] SI_2049 -> B (gold: B)
[18/203] SI_2056 -> A (gold: A)
[19/203] SI_2092 -> A (gold: A)
[20/203] SI_2097 -> A (gold: A)
[21/203] SI_2100 -> A (gold: Both)
[22/203] SI_2105 -> A (gold: A)
[23/203] SI_2111 -> B (gold: A)
[24/203] SI_2112 -> B (gold: A)
[25/203] SI_2124 -> A (gold: A)
[26/203] SI_2127 -> A (gold: A)
[27/203] SI_2129 -> A (gold: A)
[28/203] SI_2132 -> B (gold: B)
[29/203] SI_2151 -> A (gold: A)
[30/203] SI_2157 -> A (gold: A)
[31/203] SI_2158 -> A (gold: A)
[32/203] SI_21

In [ ]:
import json

with open("/content/sri_lankan_results.json", "r", encoding="utf-8") as f:
    results = json.load(f)

correct = sum(1 for item in results if item.get("model_answer") == item.get("gold_answer"))
total = len(results)

print(f"Correct: {correct} / {total}")
print(f"Accuracy: {correct / total:.2%}")

Correct: 138 / 203
Accuracy: 67.98%


#CHINESE LANGUAGE

In [ ]:
import json

DATA_PATH = "/content/sample_data/chinese_simplified.json"

with open(DATA_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Loaded {len(data)} items")

Loaded 790 items


In [ ]:
INSTRUCTION_PROMPT = (
    """
You are a Chinese Expert for answering multiple-choice questions.
You need to always evaluate these questions based on the Chinese context and provide the most accurate answer.
You should only respond with the correct option text, without any additional explanation or commentary.

Question: {question}
Option A: {option A}
Option B: {option B}
Option C: {option C}
Option D: {option D}
Return only the correct option text. Expected output is either the text of 'A', 'B', 'C', or 'D'. "Question": "

"""
)

def build_prompt(item):
    lines = [INSTRUCTION_PROMPT, "", item["scenario_question"], "", "Options:"]
    for letter, text in item["options"].items():
        lines.append(f"{letter}. {text}")
    return "\n".join(lines)

In [ ]:
import re
def generate_answer(prompt, valid_letters, max_new_tokens=20):
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True,enable_thinking=False,
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)

    input_len = inputs["input_ids"].shape[-1]
    raw = tokenizer.decode(output[0][input_len:], skip_special_tokens=True).strip()

    # First try a clean letter match (model followed instructions)
    match = re.search(r"\b([A-D])\b", raw.upper())
    if match and match.group(1) in valid_letters:
        predicted = match.group(1)
    else:
        predicted = None

    return predicted, raw

In [ ]:
OUTPUT_PATH = "/content/chinese_results.json"

for i, item in enumerate(data):
    if item.get("model_answer"):  # already processed, skip (resume-safe)
        continue

    prompt = build_prompt(item)
    predicted, raw_output = generate_answer(prompt, list(item["options"].keys()))

    item["model_answer"] = predicted
    item["raw_model_output"] = raw_output

    with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    print(f"[{i+1}/{len(data)}] {item['id']} -> {predicted} (gold: {item['gold_answer']})")

print("All done.")

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


[1/790] ZH_0_0 -> C (gold: A)
[2/790] ZH_0_3 -> C (gold: C)
[3/790] ZH_0_41 -> C (gold: A)
[4/790] ZH_0_42 -> A (gold: B)
[5/790] ZH_0_64 -> D (gold: A)
[6/790] ZH_1_47 -> A (gold: A)
[7/790] ZH_1_57 -> A (gold: A)
[8/790] ZH_1_58 -> A (gold: A)
[9/790] ZH_1_69 -> C (gold: A)
[10/790] ZH_1_77 -> A (gold: A)
[11/790] ZH_2_27 -> D (gold: A)
[12/790] ZH_2_48 -> A (gold: A)
[13/790] ZH_2_57 -> D (gold: A)
[14/790] ZH_2_60 -> D (gold: B)
[15/790] ZH_2_69 -> A (gold: A)
[16/790] ZH_3_14 -> A (gold: C)
[17/790] ZH_3_15 -> C (gold: A)
[18/790] ZH_3_16 -> B (gold: A)
[19/790] ZH_3_22 -> C (gold: C)
[20/790] ZH_3_79 -> C (gold: B)
[21/790] ZH_4_9 -> A (gold: B)
[22/790] ZH_4_23 -> A (gold: C)
[23/790] ZH_4_38 -> B (gold: D)
[24/790] ZH_4_46 -> B (gold: B)
[25/790] ZH_4_74 -> A (gold: A)
[26/790] ZH_5_14 -> A (gold: A)
[27/790] ZH_5_22 -> A (gold: A)
[28/790] ZH_5_73 -> B (gold: A)
[29/790] ZH_5_75 -> D (gold: D)
[30/790] ZH_5_76 -> A (gold: A)
[31/790] ZH_6_20 -> A (gold: A)
[32/790] ZH_6_46 -> 

In [ ]:
import json

with open("/content/chinese_results.json", "r", encoding="utf-8") as f:
    results = json.load(f)

correct = sum(1 for item in results if item.get("model_answer") == item.get("gold_answer"))
total = len(results)

print(f"Correct: {correct} / {total}")
print(f"Accuracy: {correct / total:.2%}")

Correct: 422 / 790
Accuracy: 53.42%


#INDONESIAN LANGUAGE

In [ ]:
import json

DATA_PATH = "/content/sample_data/indonesian_simplified.json"

with open(DATA_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Loaded {len(data)} items")

Loaded 366 items


In [ ]:
INSTRUCTION_PROMPT = (
    """
    You are an Indonesian Expert for answering multiple-choice questions.
    You need to always evaluate these questions based on the Indonesian context and provide the most accurate answer.
    You should only respond with the correct option text, without any additional explanation or commentary.

Question: {question}
Option A: {option A}
Option B: {option B}
Option C: {option C}
Option D: {option D}
Return only the correct option text. Expected output is either the text of 'A', 'B', 'C', or 'D'.


"""
)

def build_prompt(item):
    lines = [INSTRUCTION_PROMPT, "", item["scenario_question"], "", "Options:"]
    for letter, text in item["options"].items():
        lines.append(f"{letter}. {text}")
    return "\n".join(lines)

In [ ]:
import re
def generate_answer(prompt, valid_letters, max_new_tokens=20):
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True,enable_thinking=False,
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)

    input_len = inputs["input_ids"].shape[-1]
    raw = tokenizer.decode(output[0][input_len:], skip_special_tokens=True).strip()

    # First try a clean letter match (model followed instructions)
    match = re.search(r"\b([A-D])\b", raw.upper())
    if match and match.group(1) in valid_letters:
        predicted = match.group(1)
    else:
        predicted = None

    return predicted, raw

In [ ]:
OUTPUT_PATH = "/content/indonesian_results.json"

for i, item in enumerate(data):
    if item.get("model_answer"):  # already processed, skip (resume-safe)
        continue

    prompt = build_prompt(item)
    predicted, raw_output = generate_answer(prompt, list(item["options"].keys()))

    item["model_answer"] = predicted
    item["raw_model_output"] = raw_output

    with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    print(f"[{i+1}/{len(data)}] {item['id']} -> {predicted} (gold: {item['gold_answer']})")

print("All done.")

[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


[1/366] ID_3 -> C (gold: A)


[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


[2/366] ID_5 -> A (gold: B)


[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


...

[363/366] ID_2019 -> B (gold: B)


[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


[364/366] ID_2040 -> B (gold: B)


[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


[365/366] ID_2050 -> C (gold: C)
[366/366] ID_2058 -> C (gold: C, D)
All done.


In [ ]:
import json

with open("/content/indonesian_results.json", "r", encoding="utf-8") as f:
    results = json.load(f)

correct = 0
for item in results:
    model_answer = item.get("model_answer")
    gold_answer = item.get("gold_answer", "")

    gold_letters = [g.strip() for g in gold_answer.split(",")]  # handles single or multiple gold answers

    if model_answer is not None and model_answer.strip() in gold_letters:
        correct += 1

total = len(results)

print(f"Correct: {correct} / {total}")
print(f"Accuracy: {correct / total:.2%}")

Correct: 236 / 366
Accuracy: 64.48%
